# Cartograma de Dorling para Analise de Criminalidade em Belem (PA)
### Pipeline de Processamento Geoespacial e Interpolacao Geometrica em R

Este caderno executa o fluxo computacional em R estruturado a partir da metodologia do repositorio votemapswitzerland:
1. Carregamento vetorial: malha de bairros de Belem e reprojecao para SIRGAS 2000 UTM 22S (EPSG:31982).
2. Dados criminais: agregados do Sistema Integrado de Seguranca Publica (SIAC/SEGUP-PA).
3. Simulacao fisica de particulas (particles + tidygraph): forcas de atracao direcional e colisao com resguardo de sobreposicao.
4. Interpolacao geometrica continua (transformr + gganimate): deformacao progressiva dos limites poligonais ate os circulos proporcionais.

## 1. Instalacao de Dependencias de Sistema e Pacotes R

In [ ]:
# Instalacao de bibliotecas C/C++ de suporte geoespacial no Ubuntu
system("apt-get update -qq && apt-get install -y -qq libudunits2-dev libgdal-dev libgeos-dev libproj-dev cargo", intern = TRUE)

# Pacotes R necessarios
pacotes <- c(
  "sf", "tidyverse", "tidygraph", "particles", 
  "transformr", "gganimate", "gifski", "viridis", "scales", "stringi"
)

novos_pacotes <- pacotes[!(pacotes %in% installed.packages()[, "Package"])]
if (length(novos_pacotes) > 0) {
  install.packages(novos_pacotes, repos = "https://cloud.r-project.org", quiet = TRUE)
}

message(">> Dependencias instaladas com sucesso.")

## 2. Carga dos Modulos R

In [ ]:
source("R/01_geometrias_bairros.R")
source("R/02_dorling_fisica.R")
source("R/03_animacao_morphing.R")

message(">> Modulos R carregados no ambiente de execucao.")

## 3. Preparacao das Geometrias e Cruzamento Tabular

In [ ]:
bairros_dados <- preparar_geometrias_bairros(
  geojson_path = "data/raw/bairros_belem.geojson",
  crimes_csv_path = "data/processed/crimes_bairros_agregado.csv",
  crs_metrico = 31982
)

# Exibir primeiros registros ordenados por total de ocorrencias
bairros_dados %>%
  st_drop_geometry() %>%
  select(BAIRRO_NORM, TOTAL_CRIMES, ROUBO, FURTO, HOMICIDIO, area_km2) %>%
  arrange(desc(TOTAL_CRIMES)) %>%
  head(5)

## 4. Execucao da Simulacao Fisica (Dorling Layout)

In [ ]:
bairros_dorling <- calcular_dorling_layout(
  bairros_sf = bairros_dados,
  coluna_metrica = "TOTAL_CRIMES",
  fator_escala_k = 65,
  padding = 35,
  iteracoes = 180
)

message(">> Simulacao de forcas concluida.")

## 5. Renderizacao da Animacao (Morphing Geometrico)

In [ ]:
output_gif <- gerar_animacao_cartograma(
  bairros_dorling_sf = bairros_dorling,
  coluna_valor = "TOTAL_CRIMES",
  output_file = "outputs/animacao_dorling_seguranca_belem.gif",
  n_frames = 100,
  fps = 18,
  largura = 1100,
  altura = 850
)

# Visualizacao do arquivo gerado no caderno
knitr::include_graphics(output_gif)